# LAB DAY 19 — GraphRAG pipeline (Tech/EV Corpus)

Notebook gọi các module trong `src/` và hiển thị kết quả inline.

**Stack:** langextract (Step 1) → NetworkX (Step 2/3) → ChromaDB + MiniLM (Flat RAG) → OpenAI gpt-4o.

> Chạy bằng kernel của conda env `main`. Cần `OPENAI_API_KEY` trong `.env`.
> Các bước đã chạy qua `run.sh` sẽ được tái dùng (extract có checkpoint, không tốn token lại).

In [ ]:
import os, sys, json
os.environ['PYTHONUTF8'] = '1'
sys.path.insert(0, os.path.abspath('.'))
from IPython.display import Image, Markdown, display
from src.config import OUT_DIR
print('Output dir:', OUT_DIR)

## Step 0 — Tiền xử lý (làm sạch + bỏ doc nhị phân + gộp Query/Title/Snippet)

In [ ]:
from src import preprocess
preprocess.run()
docs = [json.loads(l) for l in open(OUT_DIR/'clean_docs.jsonl', encoding='utf-8')]
print('Số doc:', len(docs)); print('Ví dụ text head:\n', docs[1]['text'][:300])

## Step 1 — Indexing: trích xuất triple bằng langextract
Có checkpoint theo doc trong `outputs/triples_by_doc/`. Bỏ comment để chạy lại (tốn token).

In [ ]:
# from src import extract; extract.run()   # <-- bỏ comment để (re)chạy trích xuất
triples = [json.loads(l) for l in open(OUT_DIR/'triples.jsonl', encoding='utf-8')]
print('Tổng triples:', len(triples))
for t in triples[:8]:
    print(f"  ({t['subject']}) -[{t['relation']}]-> ({t['object']})")

## Step 2 — Dựng đồ thị + khử trùng lặp (NetworkX)

In [ ]:
from src import build_graph
build_graph.run()
display(Markdown('**graph_stats.json**'))
print(open(OUT_DIR/'graph_stats.json', encoding='utf-8').read())

## Deliverable 2 — Ảnh đồ thị tri thức

In [ ]:
from src import visualize
visualize.run()
display(Image(filename=str(OUT_DIR/'graph.png')))
display(Image(filename=str(OUT_DIR/'graph_subgraph.png')))

## Step 3 — Demo Flat RAG vs GraphRAG trên 1 câu hỏi multi-hop

In [ ]:
from src import flat_rag, graph_rag
G = graph_rag.load_graph()
col = flat_rag.build_index()
q = 'Những hãng nào tăng trưởng doanh số EV trên 50% YoY Q1 2024, và Tesla thay đổi thế nào?'
fa, _, fsrc = flat_rag.answer(col, q)
ga, _, gctx, ginfo = graph_rag.answer(G, q)
display(Markdown(f"**Câu hỏi:** {q}\n\n**Flat RAG:** {fa}\n\n**GraphRAG:** {ga}\n\n*Node đồ thị dùng:* {ginfo['linked_nodes']}"))
print('--- Ngữ cảnh GraphRAG (2-hop) ---\n', gctx[:1000])

## Step 4 — Benchmark 20 câu + Bảng so sánh (Deliverable 3)

In [ ]:
# from src import benchmark; benchmark.run()   # <-- bỏ comment để chạy lại 20 câu (tốn token)
display(Markdown(open(OUT_DIR/'benchmark.md', encoding='utf-8').read()))

## Deliverable 4 — Phân tích chi phí (token + thời gian)

In [ ]:
from src import cost
cost.run()
display(Markdown(open(OUT_DIR/'cost.md', encoding='utf-8').read()))